In [1]:
# Import libraries
from pathlib import Path
import re
import json
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Intelligent_Document_Processing"
)

OCR_PATH = PROJECT_ROOT / "outputs" / "ocr"

CLEANED_PATH = PROJECT_ROOT / "outputs" / "cleaned"

In [3]:
for folder in [
    CLEANED_PATH / "invoices",
    CLEANED_PATH / "resumes",
    CLEANED_PATH / "id_cards"
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("Cleaning output folders ready.")

Cleaning output folders ready.


In [4]:
# Verify OCR folders

print("Invoices :", (OCR_PATH / "invoices").exists())
print("Resumes  :", (OCR_PATH / "resumes").exists())
print("ID Cards :", (OCR_PATH / "id_cards").exists())

Invoices : True
Resumes  : True
ID Cards : True


In [5]:
# Find TXT files

def get_text_files(folder):
    """
    Return all TXT files from a folder.
    """

    folder = Path(folder)

    return sorted(
        file
        for file in folder.glob("*.txt")
        if file.is_file()
    )

In [6]:
invoice_files = get_text_files(
    OCR_PATH / "invoices"
)

resume_files = get_text_files(
    OCR_PATH / "resumes"
)

id_card_files = get_text_files(
    OCR_PATH / "id_cards"
)

print("Invoice OCR files :", len(invoice_files))
print("Resume OCR files  :", len(resume_files))
print("ID Card OCR files :", len(id_card_files))

Invoice OCR files : 3
Resume OCR files  : 2
ID Card OCR files : 3


In [7]:
# Read OCR text

def read_text_file(file_path):
    """
    Read a UTF-8 text file.
    """

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        return file.read()

In [8]:
sample_invoice = invoice_files[0]

raw_invoice_text = read_text_file(
    sample_invoice
)

print(raw_invoice_text)

tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


In [9]:
# Unicode normalization

def normalize_unicode(text):
    """
    Normalize Unicode characters.
    """

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    return text

In [10]:
text = normalize_unicode(
    raw_invoice_text
)

print(text)

tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


In [11]:
# Normalize line breaks

def normalize_line_breaks(text):
    """
    Normalize different newline formats.
    """

    text = text.replace(
        "\r\n",
        "\n"
    )

    text = text.replace(
        "\r",
        "\n"
    )

    return text

In [12]:
def clean_characters(text):
    """
    Remove control characters while preserving
    useful punctuation.
    """

    cleaned = []

    for char in text:

        category = unicodedata.category(char)

        if category.startswith("C"):
            if char in ["\n", "\t"]:
                cleaned.append(char)
        else:
            cleaned.append(char)

    return "".join(cleaned)

In [13]:
# Normalize whitespace

def normalize_whitespace(text):
    """
    Normalize spaces and tabs.
    """

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

In [14]:
# Remove empty lines

def remove_empty_lines(text):
    """
    Remove unnecessary blank lines.
    """

    lines = text.splitlines()

    cleaned_lines = [
        line.strip()
        for line in lines
        if line.strip()
    ]

    return "\n".join(
        cleaned_lines
    )

In [15]:
# Remove duplicate lines

def remove_duplicate_lines(text):
    """
    Remove duplicate lines while preserving order.
    """

    lines = text.splitlines()

    seen = set()
    result = []

    for line in lines:

        normalized = line.lower().strip()

        if normalized not in seen:

            seen.add(normalized)
            result.append(line)

    return "\n".join(result)

In [16]:
# Normalize email addresses

def normalize_emails(text):
    """
    Normalize whitespace around email addresses.
    """

    pattern = r'[\w\.-]+@[\w\.-]+\.\w+'

    emails = re.findall(
        pattern,
        text
    )

    for email in emails:

        normalized = email.lower().strip()

        text = text.replace(
            email,
            normalized
        )

    return text

In [22]:
def normalize_phone_numbers(text):
    """
    Normalize common phone number spacing.
    """

    text = re.sub(
        r'(\+91)\s+',
        r'\1 ',
        text
    )

    text = re.sub(
        r'(\d{5})\s+(\d{5})',
        r'\1 \2',
        text
    )

    return text

In [17]:
def normalize_emails(text):
    """
    Normalize whitespace around email addresses.
    """

    pattern = r'[\w\.-]+@[\w\.-]+\.\w+'

    emails = re.findall(
        pattern,
        text
    )

    for email in emails:

        normalized = email.lower().strip()

        text = text.replace(
            email,
            normalized
        )

    return text

In [18]:
def normalize_dates(text):
    """
    Normalize common date separators.
    """

    text = re.sub(
        r'(\d{1,2})[.-](\d{1,2})[.-](\d{2,4})',
        r'\1/\2/\3',
        text
    )

    return text

In [19]:
def normalize_amounts(text):
    """
    Normalize common currency spacing.
    """

    text = re.sub(
        r'₹\s+',
        '₹',
        text
    )

    text = re.sub(
        r'\$\s+',
        '$',
        text
    )

    return text

In [20]:
def clean_text(text):
    """
    Complete general OCR text-cleaning pipeline.
    """

    text = normalize_unicode(text)

    text = normalize_line_breaks(text)

    text = clean_characters(text)

    text = normalize_whitespace(text)

    text = remove_empty_lines(text)

    text = remove_duplicate_lines(text)

    text = normalize_emails(text)

    text = normalize_phone_numbers(text)

    text = normalize_dates(text)

    text = normalize_amounts(text)

    text = normalize_whitespace(text)

    return text.strip()

In [23]:
cleaned_invoice = clean_text(
    raw_invoice_text
)

print(cleaned_invoice)

tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


In [24]:
print("RAW TEXT")
print("=" * 60)
print(raw_invoice_text)

print("\n\nCLEANED TEXT")
print("=" * 60)
print(cleaned_invoice)

RAW TEXT
tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


CLEANED TEXT
tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


In [25]:
def clean_document_text(
    text,
    document_type
):
    """
    Apply general and document-specific
    text cleaning.
    """

    text = clean_text(text)

    if document_type == "invoice":

        # Normalize common invoice separators
        text = re.sub(
            r'\s*:\s*',
            ': ',
            text
        )

    elif document_type == "resume":

        # Preserve resume sections
        text = re.sub(
            r'[ \t]+',
            ' ',
            text
        )

    elif document_type == "id_card":

        # Keep compact ID-card text
        text = re.sub(
            r'\n{2,}',
            '\n',
            text
        )

    return text.strip()

In [26]:
# Batch cleaning

def batch_clean(
    input_folder,
    output_folder,
    document_type
):
    """
    Clean all OCR TXT files in a folder.
    """

    input_folder = Path(
        input_folder
    )

    output_folder = Path(
        output_folder
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    files = get_text_files(
        input_folder
    )

    results = {}

    for file_path in files:

        try:

            raw_text = read_text_file(
                file_path
            )

            cleaned = clean_document_text(
                raw_text,
                document_type
            )

            results[file_path.name] = {
                "raw_text": raw_text,
                "cleaned_text": cleaned,
                "raw_characters": len(raw_text),
                "cleaned_characters": len(cleaned),
                "raw_words": len(raw_text.split()),
                "cleaned_words": len(cleaned.split())
            }

            output_file = (
                output_folder /
                file_path.name
            )

            with open(
                output_file,
                "w",
                encoding="utf-8"
            ) as file:

                file.write(cleaned)

        except Exception as e:

            print(
                f"Error processing "
                f"{file_path.name}: {e}"
            )

    return results

In [27]:
# Clean invoices

invoice_cleaned = batch_clean(
    OCR_PATH / "invoices",
    CLEANED_PATH / "invoices",
    "invoice"
)

In [28]:
# Clean resumes

resume_cleaned = batch_clean(
    OCR_PATH / "resumes",
    CLEANED_PATH / "resumes",
    "resume"
)

In [29]:
# Clean ID cards

id_card_cleaned = batch_clean(
    OCR_PATH / "id_cards",
    CLEANED_PATH / "id_cards",
    "id_card"
)

In [30]:
# Create cleaning summary

def create_cleaning_summary(
    results,
    document_type
):

    rows = []

    for filename, data in results.items():

        rows.append({
            "document_type": document_type,
            "filename": filename,
            "raw_characters":
                data["raw_characters"],
            "cleaned_characters":
                data["cleaned_characters"],
            "raw_words":
                data["raw_words"],
            "cleaned_words":
                data["cleaned_words"]
        })

    return pd.DataFrame(rows)

In [31]:
invoice_summary = create_cleaning_summary(
    invoice_cleaned,
    "Invoice"
)

resume_summary = create_cleaning_summary(
    resume_cleaned,
    "Resume"
)

id_summary = create_cleaning_summary(
    id_card_cleaned,
    "ID Card"
)

In [32]:
cleaning_summary = pd.concat(
    [
        invoice_summary,
        resume_summary,
        id_summary
    ],
    ignore_index=True
)

cleaning_summary

,document_type,filename,raw_characters,cleaned_characters,raw_words,cleaned_words
0,Invoice,X00016469612.txt,389,390,74,75
1,Invoice,X00016469620.txt,708,679,136,126
2,Invoice,X00016469622.txt,472,450,78,73
3,Resume,Image_1.txt,157,157,23,23
4,Resume,Image_10.txt,42,42,8,8
5,ID Card,0005_jpeg_jpg.rf.252590ac8ae7f0f84a08f15a164d6...,127,127,16,16
6,ID Card,0521_adhar_jpg.rf.0188ba6593c38a70e56de2509a24...,31,16,7,4
7,ID Card,0521_adhar_jpg.rf.0b52a68eba1d0f3ea5d868a66390...,36,21,9,6


In [33]:
# Save summary

cleaning_summary.to_csv(
    CLEANED_PATH / "cleaning_summary.csv",
    index=False
)

print("Cleaning summary saved.")

Cleaning summary saved.


In [34]:
# Save JSON

def save_cleaning_json(
    results,
    output_file
):

    output_file = Path(
        output_file
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            results,
            file,
            indent=4,
            ensure_ascii=False
        )

In [35]:
save_cleaning_json(
    invoice_cleaned,
    CLEANED_PATH / "invoice_cleaned.json"
)

save_cleaning_json(
    resume_cleaned,
    CLEANED_PATH / "resume_cleaned.json"
)

save_cleaning_json(
    id_card_cleaned,
    CLEANED_PATH / "id_card_cleaned.json"
)

In [36]:
# Compare Raw vs Cleaned

sample_file = list(
    invoice_cleaned.keys()
)[0]

data = invoice_cleaned[
    sample_file
]

print("RAW OCR TEXT")
print("=" * 70)
print(data["raw_text"])

print("\n\nCLEANED TEXT")
print("=" * 70)
print(data["cleaned_text"])

RAW OCR TEXT
tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour :ling Acljustnienl;
0.00
Round:d Total (RM):
Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN


CLEANED TEXT
tan woon yann
BOOK TA _K (TAMAN DAYA) SDN BHD
789H17-W
NO.S: 55,57 & S9, JALAN SAGU I8,
TAMAN DAYA,
81100 JOIIOR BAHRU,
JOHOR
HHE
Dociinent No
Dale
25/12p2018 8: 13.39 PM
Cashier
MANIS
Menxler
CASH BILL
CODE/DESc
Disc
Abl( J J!l
QTY
Ri
RM
1 PC
9.00)
0,C0
9.00
Total
S
Rour: ling Acljustnienl;
0.00
Round: d Total (RM): Cash
CHIANGE
0o
EXCH INNIGEABLE
# #ii i &
THANK YOU
PLEASE COI 'E AGATN
